In [ ]:
!pip install timm -q

import torch
import torch.nn as nn
import torch.optim as optim
import timm
import pandas as pd
import numpy as np
import json
import os
import random
from PIL import Image, ImageEnhance
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    f1_score, accuracy_score,
    mean_absolute_error
)
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

drive.mount('/content/drive')

BASE          = 'YOUR_DRIVE_PATH'
HUMAN_FOLDER  = f'{BASE}/images_human_fixed'
AUG_FOLDER    = f'{BASE}/images_human_aug'
RESULT_FOLDER = f'{BASE}/results'
MODEL_FOLDER  = f'{BASE}/models_human_balanced'

os.makedirs(MODEL_FOLDER,  exist_ok=True)
os.makedirs(RESULT_FOLDER, exist_ok=True)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
CUTOFF = 6
print(f'Device: {device}')

In [ ]:
train_df = pd.read_csv(
    f'{BASE}/labels/human_train_balanced.csv'
)
val_df   = pd.read_csv(
    f'{BASE}/labels/human_val.csv'
)
test_df  = pd.read_csv(
    f'{BASE}/labels/human_test.csv'
)

for d in [train_df, val_df, test_df]:
    d['dementia_risk'] = d['dementia_risk'].astype(bool)

print(f'Train (balanced): {len(train_df)} รูป')
print(f'  Normal  : {(~train_df["dementia_risk"]).sum()}')
print(f'  Dementia: {train_df["dementia_risk"].sum()}')
print(f'Val : {len(val_df)} รูป')
print(f'Test: {len(test_df)} รูป')

domain_cols = {
    'domain_A': 'A: digit_count',
    'domain_B': 'B: worst_quadrant',
    'domain_C': 'C: spatial',
    'domain_D': 'D: hands_present',
    'domain_E': 'E: hands_placement',
}

print(f'\n{"Domain":<25} {"C0":>8} '
      f'{"C1":>8} {"C2":>8} {"ratio C2":>10}')
print('-'*58)

for col, name in domain_cols.items():
    c0    = (train_df[col] == 0).sum()
    c1    = (train_df[col] == 1).sum()
    c2    = (train_df[col] == 2).sum()
    total = c0 + c1 + c2
    print(f'{name:<25} {c0:>8} '
          f'{c1:>8} {c2:>8} '
          f'{c2/total*100:>9.1f}%')

In [ ]:
TARGET_DEMENTIA = 800

dementia_df = train_df[train_df['dementia_risk']]
normal_df   = train_df[~train_df['dementia_risk']]

need_more = max(
    0, TARGET_DEMENTIA - len(dementia_df)
)
print(f'Dementia : {len(dementia_df)} รูป')
print(f'เพิ่ม      : {need_more} รูป')

aug_rows  = []
generated = 0

if need_more > 0:
    print('\nกำลัง augment...')
    while generated < need_more:
        for _, row in dementia_df.iterrows():
            if generated >= need_more:
                break
            try:
                src = (f'{HUMAN_FOLDER}/'
                       f'{row["filename"]}')
                img = Image.open(src).convert('RGB')

                factor = random.uniform(0.8, 1.2)
                img    = ImageEnhance.Brightness(
                    img
                ).enhance(factor)
                factor = random.uniform(0.8, 1.2)
                img    = ImageEnhance.Contrast(
                    img
                ).enhance(factor)

                new_name = (
                    f'aug_n_{generated:05d}_'
                    f'{row["filename"]}'
                )
                img.save(
                    f'{AUG_FOLDER}/{new_name}',
                    'JPEG', quality=95
                )

                new_row             = row.copy()
                new_row['filename'] = new_name
                aug_rows.append(new_row)
                generated += 1
            except:
                continue

    aug_df      = pd.DataFrame(aug_rows)
    train_df    = pd.concat(
        [train_df, aug_df], ignore_index=True
    ).sample(frac=1, random_state=42).reset_index(
        drop=True
    )

print(f'\nTrain set :')
print(f'  Normal  : {(~train_df["dementia_risk"]).sum()}')
print(f'  Dementia: {train_df["dementia_risk"].sum()}')
print(f'  Total   : {len(train_df)}')

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch

class CDTDataset(Dataset):
    def __init__(self, df, image_folder,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']

        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'

        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224),
                           (255, 255, 255))

        if self.transform:
            img = self.transform(img)

        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)

        return img, labels

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

In [ ]:
import torch.nn as nn
import timm
from torchvision import models

class CDTModel(nn.Module):
    def __init__(self, backbone_name='resnet50',
                 num_domains=5, num_classes=3):
        super().__init__()

        if backbone_name == 'resnet50':
            backbone = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1
            )
            self.feature_dim = 2048
            for param in backbone.parameters():
                param.requires_grad = False
            self.backbone = nn.Sequential(
                *list(backbone.children())[:-1]
            )
        elif backbone_name == 'vit':
            self.backbone = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True,
                num_classes=0
            )
            self.feature_dim = 768
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.backbone_name = backbone_name
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.heads_cls = nn.ModuleList([
            nn.Linear(512, num_classes)
            for _ in range(num_domains)
        ])
        self.heads_reg = nn.ModuleList([
            nn.Linear(512, 1)
            for _ in range(num_domains)
        ])

    def forward(self, x, mode='reg'):
        if self.backbone_name == 'resnet50':
            feat = self.backbone(x)
        else:
            feat = self.backbone(x)
            feat = feat.unsqueeze(-1).unsqueeze(-1)
        feat = self.shared(feat)
        if mode == 'reg':
            return [head(feat).squeeze(-1)
                    for head in self.heads_reg]
        else:
            return [head(feat)
                    for head in self.heads_cls]

In [ ]:
import torch.optim as optim
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    f1_score, accuracy_score,
    mean_absolute_error, mean_squared_error
)

DOMAIN_WEIGHTS = torch.tensor(
    [1.0, 1.5, 1.0, 1.0, 1.0]
).to(device)

def compute_loss(outputs, labels, mode='reg'):
    total = 0
    for i, output in enumerate(outputs):
        if mode == 'reg':
            loss = nn.MSELoss()(
                output, labels[:, i].float()
            )
        else:
            loss = nn.CrossEntropyLoss()(
                output, labels[:, i].long()
            )
        total += loss * DOMAIN_WEIGHTS[i]
    return total / DOMAIN_WEIGHTS.sum()

def get_predictions(outputs, mode='reg'):
    preds = []
    for output in outputs:
        if mode == 'reg':
            pred = output.round().clamp(0, 2).long()
        else:
            pred = output.argmax(dim=1)
        preds.append(pred)
    return torch.stack(preds, dim=1)

In [ ]:
def evaluate_model(model, loader, mode='reg'):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device)
            outputs = model(images, mode=mode)
            preds   = get_predictions(outputs, mode)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    pred_total    = all_preds.sum(axis=1)
    true_total    = all_labels.sum(axis=1)
    pred_dementia = (pred_total < CUTOFF).astype(int)
    true_dementia = (true_total < CUTOFF).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_dementia, pred_dementia, labels=[0,1]
    ).ravel()

    sens = tp/(tp+fn) if (tp+fn) > 0 else 0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0
    try:
        auc = roc_auc_score(true_dementia, -pred_total)
    except:
        auc = 0

    domain_metrics = {}
    for i, d in enumerate(['A','B','C','D','E']):
        domain_metrics[d] = {
            'acc': accuracy_score(
                all_labels[:,i], all_preds[:,i]),
            'mae': mean_absolute_error(
                all_labels[:,i], all_preds[:,i]),
            'f1' : f1_score(
                all_labels[:,i], all_preds[:,i],
                average='macro', zero_division=0),
        }

    return {
        'sensitivity'  : sens,
        'specificity'  : spec,
        'auc'          : auc,
        'f1'           : f1_score(
            true_dementia, pred_dementia,
            zero_division=0),
        'domain_metrics': domain_metrics,
        'avg_acc'      : np.mean([
            m['acc']
            for m in domain_metrics.values()]),
        'pred_total'   : pred_total,
        'true_total'   : true_total,
        'true_dementia': true_dementia,
        'all_preds'    : all_preds,
        'all_labels'   : all_labels,
    }

In [ ]:
def train_model(backbone, mode, model_name,
                epochs=20, patience=10):

    print(f'\n{"="*60}')
    print(f'Training: {model_name}')
    print(f'Backbone: {backbone} | Loss: {mode}')
    print(f'Balanced Train (40% downsample)')
    print(f'{"="*60}')

    train_ds = CDTDataset(
        train_df, HUMAN_FOLDER,
        AUG_FOLDER, train_transform
    )
    val_ds = CDTDataset(
        val_df, HUMAN_FOLDER,
        None, val_transform
    )
    test_ds = CDTDataset(
        test_df, HUMAN_FOLDER,
        None, val_transform
    )

    train_loader = DataLoader(
        train_ds, batch_size=16,
        shuffle=True, num_workers=2
    )
    val_loader = DataLoader(
        val_ds, batch_size=16,
        shuffle=False, num_workers=2
    )
    test_loader_local = DataLoader(
        test_ds, batch_size=16,
        shuffle=False, num_workers=2
    )

    model     = CDTModel(backbone_name=backbone).to(device)
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad,
               model.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    best_combined = -1
    best_state    = None
    no_improve    = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'sensitivity': [], 'specificity': [],
        'auc': []
    }

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            outputs = model(images, mode=mode)
            loss    = compute_loss(outputs, labels, mode)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        val_results = evaluate_model(
            model, val_loader, mode=mode
        )

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = model(images, mode=mode)
                loss    = compute_loss(
                    outputs, labels, mode
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['sensitivity'].append(
            val_results['sensitivity'])
        history['specificity'].append(
            val_results['specificity'])
        history['auc'].append(val_results['auc'])

        combined = (
            val_results['sensitivity'] +
            val_results['specificity']
        ) / 2
        if combined > best_combined:
            best_combined = combined
            best_state    = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stop epoch {epoch+1}')
                break

        print(f'Epoch {epoch+1:2d}/{epochs} | '
              f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
              f'Sens: {val_results["sensitivity"]:.3f} | '
              f'Spec: {val_results["specificity"]:.3f} | '
              f'AUC: {val_results["auc"]:.3f}')

    model.load_state_dict(best_state)
    torch.save(
        best_state,
        f'{MODEL_FOLDER}/{model_name}_best.pth'
    )
    with open(
        f'{RESULT_FOLDER}/history_{model_name}.json',
        'w'
    ) as f:
        json.dump(history, f)

    test_results = evaluate_model(
        model, test_loader_local, mode=mode
    )
    return model, history, test_results

In [ ]:
results_balanced = {}

for backbone, mode in [
    ('resnet50', 'reg'),
    ('resnet50', 'cls'),
    ('vit',      'reg'),
    ('vit',      'cls'),
]:
    epochs     = 15 if backbone == 'vit' else 20
    model_name = f'balanced_{backbone}_{mode}'
    path       = f'{MODEL_FOLDER}/{model_name}_best.pth'

    if os.path.exists(path):
        print(f'\nโหลด {model_name}')
        model = CDTModel(backbone_name=backbone).to(device)
        model.load_state_dict(
            torch.load(path, map_location=device)
        )
        model.eval()

        test_ds = CDTDataset(
            test_df, HUMAN_FOLDER,
            None, val_transform
        )
        test_loader_eval = DataLoader(
            test_ds, batch_size=16, shuffle=False
        )
        test_results = evaluate_model(
            model, test_loader_eval, mode=mode
        )
    else:
        _, _, test_results = train_model(
            backbone, mode, model_name,
            epochs=epochs, patience=10
        )

    results_balanced[f'{backbone}_{mode}'] = test_results

    print(f'\n📊 {model_name}:')
    print(f'  Sens: {test_results["sensitivity"]:.3f} '
          f'{"✅" if test_results["sensitivity"]>=0.88 else "⚠️"}')
    print(f'  Spec: {test_results["specificity"]:.3f} '
          f'{"✅" if test_results["specificity"]>=0.82 else "⚠️"}')
    print(f'  AUC : {test_results["auc"]:.3f} '
          f'{"✅" if test_results["auc"]>=0.91 else "⚠️"}')

In [ ]:
previous = {
    '07k ViT REG'   : {'sens':0.751,'spec':0.844,'auc':0.882},
    '07l Hybrid VQA': {'sens':0.760,'spec':0.850,'auc':0.887},
}

print('='*80)
print('COMPARISON: Previous vs 07n Balanced')
print('='*80)
print(f'{"Model":<25} {"Sens":>10} '
      f'{"Spec":>10} {"AUC":>10}')
print('='*80)

for name, r in previous.items():
    print(f'{name:<25} '
          f'{r["sens"]:>10.3f} '
          f'{r["spec"]:>10.3f} '
          f'{r["auc"]:>10.3f}')

print('-'*80)

best_auc = max(
    r['auc'] for r in results_balanced.values()
)
for key, r in results_balanced.items():
    name = f'07n {key}'
    mark = '✅' if r['auc'] == best_auc else ''
    pass_both = '🎯' if (
        r['sensitivity'] >= 0.88 and
        r['specificity'] >= 0.82
    ) else ''
    print(f'{name:<25} '
          f'{r["sensitivity"]:>10.3f} '
          f'{r["specificity"]:>10.3f} '
          f'{r["auc"]:>10.3f} {mark}{pass_both}')

print('='*80)
print(f'{"Target":<25} {"≥0.880":>10} '
      f'{"≥0.820":>10} {"≥0.910":>10}')

with open(
    f'{RESULT_FOLDER}/07n_balanced_results.json',
    'w'
) as f:
    json.dump({
        'train_size'  : len(train_df),
        'downsample'  : '40% weight-based',
        'previous'    : previous,
        'results'     : {
            k: {
                'sensitivity': float(v['sensitivity']),
                'specificity': float(v['specificity']),
                'auc'        : float(v['auc']),
            }
            for k, v in results_balanced.items()
        }
    }, f, indent=2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from scipy.stats import gaussian_kde
import numpy as np

domain_names = {
    0: 'A: digit_count',
    1: 'B: worst_quadrant',
    2: 'C: spatial',
    3: 'D: hands_present',
    4: 'E: hands_placement',
}
colors_true = {
    0: '#3F51B5',
    1: '#00BCD4',
    2: '#8BC34A',
}

all_model_data = {}

for backbone, mode in [
    ('resnet50', 'reg'),
    ('resnet50', 'cls'),
    ('vit',      'reg'),
    ('vit',      'cls'),
]:
    model_name = f'balanced_{backbone}_{mode}'
    path       = f'{MODEL_FOLDER}/{model_name}_best.pth'

    model = CDTModel(backbone_name=backbone).to(device)
    model.load_state_dict(
        torch.load(path, map_location=device)
    )
    model.eval()

    test_ds = CDTDataset(
        test_df, HUMAN_FOLDER, None, val_transform
    )
    test_loader_eval = DataLoader(
        test_ds, batch_size=16, shuffle=False
    )

    preds_list  = []
    labels_list = []
    raw_list    = []

    with torch.no_grad():
        for images, labels in test_loader_eval:
            images  = images.to(device)
            outputs = model(images, mode=mode)
            preds   = get_predictions(outputs, mode)
            raw     = torch.stack(
                [o.squeeze(-1) if mode == 'reg'
                 else (torch.softmax(o, dim=1) *
                       torch.tensor([0.,1.,2.]).to(device)
                       ).sum(dim=1)
                 for o in outputs], dim=1
            )
            preds_list.append(preds.cpu())
            labels_list.append(labels.cpu())
            raw_list.append(raw.cpu())

    all_model_data[f'{backbone}_{mode}'] = {
        'preds' : torch.cat(preds_list).numpy(),
        'labels': torch.cat(labels_list).numpy(),
        'raw'   : torch.cat(raw_list).numpy(),
    }
    print(f' {model_name}')

In [ ]:
short_names = {
    'resnet50_reg': 'ResNet50 REG',
    'resnet50_cls': 'ResNet50 CLS',
    'vit_reg'     : 'ViT REG',
    'vit_cls'     : 'ViT CLS',
}

for m_key, data in all_model_data.items():
    preds  = data['preds']
    labels = data['labels']

    fig, axes = plt.subplots(2, 5, figsize=(22, 9))

    for d_idx, d_name in domain_names.items():
        true_d = labels[:, d_idx].astype(int)
        pred_d = preds[:, d_idx].astype(int)

        cm      = confusion_matrix(
            true_d, pred_d, labels=[0,1,2]
        )
        cm_norm = cm.astype(float)
        row_sum = cm.sum(axis=1, keepdims=True)
        row_sum[row_sum == 0] = 1
        cm_norm = cm_norm / row_sum

        ax_raw  = axes[0][d_idx]
        ax_norm = axes[1][d_idx]

        sns.heatmap(
            cm, annot=True, fmt='d',
            cmap='Blues', ax=ax_raw,
            xticklabels=['0','1','2'],
            yticklabels=['0','1','2'],
            cbar=False
        )
        acc = (true_d == pred_d).mean()
        ax_raw.set_title(
            f'{d_name}\nAcc={acc:.3f}',
            fontsize=9, fontweight='bold'
        )
        ax_raw.set_xlabel('Predicted')
        ax_raw.set_ylabel('True')

        sns.heatmap(
            cm_norm, annot=True, fmt='.2f',
            cmap='Blues', ax=ax_norm,
            xticklabels=['0','1','2'],
            yticklabels=['0','1','2'],
            cbar=False, vmin=0, vmax=1
        )
        ax_norm.set_title(
            f'{d_name}\n(normalized)',
            fontsize=9
        )
        ax_norm.set_xlabel('Predicted')
        ax_norm.set_ylabel('True')

    plt.suptitle(
        f'Confusion Matrix — 07n {short_names[m_key]}\n'
        f'Balanced Train (40% downsample)',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig(
        f'{RESULT_FOLDER}/07n_{m_key}_confusion.png',
        dpi=150, bbox_inches='tight'
    )
    plt.show()

In [ ]:
vit_path_07k = (
    f'{BASE}/models_human_undersample/'
    f'human_us_vit_reg_best.pth'
)
vit_07k = CDTModel(backbone_name='vit').to(device)
vit_07k.load_state_dict(
    torch.load(vit_path_07k, map_location=device)
)
vit_07k.eval()

test_ds_plain = CDTDataset(
    test_df, HUMAN_FOLDER, None, val_transform
)
loader_plain = DataLoader(
    test_ds_plain, batch_size=16, shuffle=False
)

raw_07k    = []
labels_07k = []
with torch.no_grad():
    for images, labels in loader_plain:
        images  = images.to(device)
        outputs = vit_07k(images, mode='reg')
        raw     = torch.stack(
            [o.squeeze(-1) for o in outputs], dim=1
        )
        raw_07k.append(raw.cpu())
        labels_07k.append(labels.cpu())

raw_07k    = torch.cat(raw_07k).numpy()
labels_07k = torch.cat(labels_07k).numpy()

fig, axes = plt.subplots(
    2, 5, figsize=(25, 10)
)

compare_data = [
    ('07k ViT REG',  raw_07k, labels_07k),
    ('07n ViT REG',
     all_model_data['vit_reg']['raw'],
     all_model_data['vit_reg']['labels']),
]

for row_idx, (name, raw, labels) in enumerate(
    compare_data
):
    for d_idx, d_name in domain_names.items():
        ax     = axes[row_idx][d_idx]
        true_d = labels[:, d_idx].astype(int)
        score_d= raw[:, d_idx]

        for true_val in [0, 1, 2]:
            mask   = (true_d == true_val)
            scores = score_d[mask]
            n      = mask.sum()
            if n < 2:
                continue
            try:
                kde    = gaussian_kde(
                    scores, bw_method=0.3
                )
                x_min  = scores.min() - 0.2
                x_max  = scores.max() + 0.2
                x_vals = np.linspace(x_min, x_max, 300)
                y_vals = kde(x_vals)
                ax.plot(
                    x_vals, y_vals,
                    color=colors_true[true_val],
                    linewidth=2,
                    label=f'true={true_val} (n={n})'
                )
                ax.fill_between(
                    x_vals, y_vals,
                    color=colors_true[true_val],
                    alpha=0.15
                )
            except:
                pass

        for x in [0.5, 1.5]:
            ax.axvline(x=x, color='gray',
                      linestyle='--', alpha=0.4)

        acc = (true_d == labels[:, d_idx].astype(int)
               ).mean() if len(true_d) else 0

        ax.set_title(
            f'{name}\n{d_name}',
            fontsize=9,
            fontweight='bold' if row_idx==1 else 'normal'
        )
        ax.set_xlabel('Score', fontsize=7)
        ax.set_ylabel('Density', fontsize=7)

        if d_idx == 0 and row_idx == 0:
            ax.legend(fontsize=7)

plt.suptitle(
    'Distribution Comparison: 07k vs 07n\n'
    'Row 1 = 07k ViT REG | Row 2 = 07n ViT REG\n'
    'blue=y_true=0  cyan=y_true=1  green=y_true=2',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/07n_vs_07k_distribution.png',
    dpi=150, bbox_inches='tight'
)
plt.show()